In [1]:
"""
Federated Learning — Personalized FL (Per-FedAvg style) + Differential Privacy
Dataset: ULB Credit Card Fraud (creditcard.csv from Kaggle)
Each bank fine-tunes the global model locally after each round.
DP-SGD noise added to gradients before aggregation.
Reports: AUC, Precision@K, Recall@1%FPR, fairness σ_AUC
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, roc_curve,
                             precision_score, recall_score, f1_score)
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import copy, warnings
warnings.filterwarnings("ignore")

# ── CONFIG ────────────────────────────────────────────────────────────────────
N_BANKS          = 5
FL_ROUNDS        = 20
GLOBAL_EPOCHS    = 3    # global training per round
FINETUNE_EPOCHS  = 5    # local fine-tuning per bank after aggregation
BATCH_SIZE       = 256
LR               = 0.001
FINETUNE_LR      = 0.0003

# Differential Privacy parameters
DP_ENABLED       = True
DP_NOISE_MULT    = 1.1   # Gaussian noise multiplier (σ)
DP_CLIP_NORM     = 1.0   # per-sample gradient clipping bound (C)
EPSILON_TARGET   = 4.0   # target ε for privacy budget reporting

RANDOM_STATE     = 42
DEVICE           = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# ── LOAD & SPLIT ──────────────────────────────────────────────────────────────
print("Loading dataset...")
df = pd.read_csv("/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv")
X  = df.drop("Class", axis=1).values
y  = df["Class"].values
scaler = StandardScaler()
X      = scaler.fit_transform(X)
N_FEAT = X.shape[1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

def non_iid_split(X, y, n, seed=42):
    rng        = np.random.default_rng(seed)
    fraud_idx  = np.where(y == 1)[0]
    legit_idx  = np.where(y == 0)[0]
    splits     = rng.dirichlet(np.ones(n) * 0.4) * len(fraud_idx)
    splits     = np.round(splits).astype(int)
    splits[-1] = len(fraud_idx) - splits[:-1].sum()
    banks, fi  = [], 0
    leg_per    = len(legit_idx) // n
    for i in range(n):
        fe  = fi + splits[i]
        idx = np.concatenate([fraud_idx[fi:fe], legit_idx[i*leg_per:(i+1)*leg_per]])
        rng.shuffle(idx)
        banks.append((X[idx], y[idx]))
        fi  = fe
    return banks

print(f"\nNon-IID split across {N_BANKS} banks:")
bank_data = non_iid_split(X_train, y_train, N_BANKS)
for i, (xb, yb) in enumerate(bank_data):
    print(f"  Bank {i+1}: {len(xb):>6,} records | fraud {yb.mean()*100:.2f}%")

# ── MODEL ─────────────────────────────────────────────────────────────────────
class FraudNet(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),  nn.ReLU(),
            nn.Linear(32, 1),   nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze()

def make_loader(X, y):
    return DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32),
                      torch.tensor(y, dtype=torch.float32)),
        batch_size=BATCH_SIZE, shuffle=True
    )

# ── DP-SGD GRADIENT PERTURBATION ─────────────────────────────────────────────
def add_dp_noise(model, clip_norm=DP_CLIP_NORM, noise_mult=DP_NOISE_MULT, n_samples=1):
    """Clip gradients and add calibrated Gaussian noise (simplified DP-SGD)."""
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    total_norm = total_norm ** 0.5

    clip_coef = clip_norm / max(total_norm, clip_norm)
    for p in model.parameters():
        if p.grad is not None:
            p.grad.data.mul_(clip_coef)
            noise_std = noise_mult * clip_norm / n_samples
            p.grad.data.add_(torch.randn_like(p.grad.data) * noise_std)

# ── LOCAL TRAIN ───────────────────────────────────────────────────────────────
def local_train(model, loader, epochs, lr, use_dp=False):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()
    for _ in range(epochs):
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            if use_dp:
                add_dp_noise(model, n_samples=len(X_b))
            optimizer.step()
    return model

# ── EVALUATE ─────────────────────────────────────────────────────────────────
def get_probs(model, X):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).cpu().numpy()

def evaluate_full(model, X, y, k_pct=0.005):
    prob  = get_probs(model, X)
    pred  = (prob >= 0.5).astype(int)
    auc   = roc_auc_score(y, prob)
    f1    = f1_score(y, pred, zero_division=0)
    prec  = precision_score(y, pred, zero_division=0)
    rec   = recall_score(y, pred, zero_division=0)

    # Precision @ K (top-k flagged as fraud)
    K       = max(1, int(len(y) * k_pct))
    top_k   = np.argsort(prob)[::-1][:K]
    prec_k  = y[top_k].mean()

    # Recall @ 1% FPR
    fpr, tpr, _ = roc_curve(y, prob)
    idx      = np.where(fpr <= 0.01)[0]
    rec_1fpr = tpr[idx[-1]] if len(idx) > 0 else 0.0

    return dict(auc=auc, f1=f1, precision=prec, recall=rec,
                prec_at_k=prec_k, recall_1fpr=rec_1fpr)

# ── PERSONALIZED FL TRAINING ──────────────────────────────────────────────────
print(f"\nStarting Personalized FL "
      f"{'+ DP (ε≈' + str(EPSILON_TARGET) + ')' if DP_ENABLED else ''}: "
      f"{FL_ROUNDS} rounds\n")

global_model  = FraudNet(N_FEAT).to(DEVICE)
# Each bank keeps its own personalized model
personal_models = [copy.deepcopy(global_model) for _ in range(N_BANKS)]

for rnd in range(1, FL_ROUNDS + 1):
    local_weights = []
    local_sizes   = []

    for bank_id, (X_b, y_b) in enumerate(bank_data):
        try:
            sm      = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(3, int(y_b.sum())-1))
            X_r, y_r = sm.fit_resample(X_b, y_b)
        except Exception:
            X_r, y_r = X_b, y_b

        # Start from global model, train locally
        local_model = copy.deepcopy(global_model)
        loader      = make_loader(X_r, y_r)
        local_model = local_train(local_model, loader, GLOBAL_EPOCHS, LR,
                                  use_dp=DP_ENABLED)

        local_weights.append(copy.deepcopy(local_model.state_dict()))
        local_sizes.append(len(X_b))

    # Weighted FedAvg aggregation
    total     = sum(local_sizes)
    new_state = copy.deepcopy(local_weights[0])
    for key in new_state:
        new_state[key] = sum(
            local_weights[i][key] * (local_sizes[i] / total)
            for i in range(N_BANKS)
        )
    global_model.load_state_dict(new_state)

    # ── PERSONALIZATION: Each bank fine-tunes the global model locally ────────
    for bank_id, (X_b, y_b) in enumerate(bank_data):
        try:
            sm      = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(3, int(y_b.sum())-1))
            X_r, y_r = sm.fit_resample(X_b, y_b)
        except Exception:
            X_r, y_r = X_b, y_b
        personal_models[bank_id] = copy.deepcopy(global_model)
        loader = make_loader(X_r, y_r)
        personal_models[bank_id] = local_train(
            personal_models[bank_id], loader, FINETUNE_EPOCHS, FINETUNE_LR,
            use_dp=False  # no DP on local fine-tuning
        )

    if rnd % 5 == 0 or rnd == 1:
        m = evaluate_full(global_model, X_test, y_test)
        print(f"  Round {rnd:>3} | AUC: {m['auc']:.4f} | F1: {m['f1']:.4f} "
              f"| Recall@1%FPR: {m['recall_1fpr']:.4f} | P@K: {m['prec_at_k']:.4f}")

# ── FINAL RESULTS ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)

print("\n[Global Model]")
m = evaluate_full(global_model, X_test, y_test)
for k, v in m.items():
    print(f"  {k:<20}: {v:.4f}")

print("\n[Per-Bank Personalized Models — Fairness Evaluation]")
bank_aucs = []
for i, (X_b, y_b) in enumerate(bank_data):
    if len(np.unique(y_b)) < 2: continue
    pm   = personal_models[i]
    m_b  = evaluate_full(pm, X_b, y_b)
    bank_aucs.append(m_b['auc'])
    print(f"  Bank {i+1}: AUC={m_b['auc']:.4f} | F1={m_b['f1']:.4f} "
          f"| Recall@1%FPR={m_b['recall_1fpr']:.4f}")

sigma = np.std(bank_aucs)
jfi   = sum(bank_aucs)**2 / (len(bank_aucs) * sum(a**2 for a in bank_aucs))
print(f"\n  σ_AUC (inter-bank fairness) : {sigma:.4f}  (lower = fairer)")
print(f"  Jain Fairness Index          : {jfi:.4f}  (closer to 1 = fairer)")

if DP_ENABLED:
    print(f"\n[Privacy Budget (Approximate)]")
    print(f"  DP noise multiplier σ = {DP_NOISE_MULT}")
    print(f"  Gradient clip norm  C = {DP_CLIP_NORM}")
    print(f"  Target ε (budget)   ≈ {EPSILON_TARGET} (use Rényi DP accountant for exact)")
    print(f"  Interpretation: ε=4 is considered strong privacy for financial FL")

Device: cuda
Loading dataset...

Non-IID split across 5 banks:
  Bank 1: 45,599 records | fraud 0.24%
  Bank 2: 45,672 records | fraud 0.40%
  Bank 3: 45,490 records | fraud 0.00%
  Bank 4: 45,592 records | fraud 0.22%
  Bank 5: 45,491 records | fraud 0.00%

Starting Personalized FL + DP (ε≈4.0): 20 rounds

  Round   1 | AUC: 0.9787 | F1: 0.6983 | Recall@1%FPR: 0.8776 | P@K: 0.2993
  Round   5 | AUC: 0.9758 | F1: 0.6975 | Recall@1%FPR: 0.8878 | P@K: 0.3028
  Round  10 | AUC: 0.9774 | F1: 0.7421 | Recall@1%FPR: 0.8878 | P@K: 0.3063
  Round  15 | AUC: 0.9778 | F1: 0.7556 | Recall@1%FPR: 0.9082 | P@K: 0.3099
  Round  20 | AUC: 0.9761 | F1: 0.7589 | Recall@1%FPR: 0.9082 | P@K: 0.3063

FINAL RESULTS

[Global Model]
  auc                 : 0.9761
  f1                  : 0.7589
  precision           : 0.6746
  recall              : 0.8673
  prec_at_k           : 0.3063
  recall_1fpr         : 0.9082

[Per-Bank Personalized Models — Fairness Evaluation]
  Bank 1: AUC=1.0000 | F1=0.9316 | Recal